# Local GW distortion for a non-isometric match

This notebook generates `fig:gromov-nonisometric-distortion`.  After computing a GW coupling between a source shape and a deformed target, we display both the correspondence and the pairwise-distance residual
$$
    |d_X(x_i,x_{i'})-d_Y(y_{\sigma(i)},y_{\sigma(i')})|,
$$
which is the local contribution minimized by the GW objective for the displayed hard correspondence.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")
for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from PIL import Image
import ot
from scipy.linalg import expm, solve
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter
from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY, DIRAC_MARKER_SIZE,
    setup_matplotlib, figure_dir, save_pdf, remove_axes, box_axes,
    interp_color, draw_point_clouds, draw_transport_segments, padded_limits,
)
setup_matplotlib()
rng = np.random.default_rng(2027)

In [2]:
NAME = "gromov-nonisometric-distortion"
out = figure_dir(NAME)
def base_shape(n=16):
    t=np.linspace(0,1,n)
    x=0.92*np.cos(1.55*np.pi*t)+0.13*t
    y=0.74*np.sin(1.55*np.pi*t)-0.28*t
    q=np.c_[x,y]
    q += 0.018*rng.normal(size=q.shape)
    return q
def rotate(q, a):
    c,s=np.cos(a),np.sin(a); R=np.array([[c,-s],[s,c]])
    return q@R.T
def gw_plan(x,y):
    C1=ot.dist(x,x); C2=ot.dist(y,y); C1/=C1.max(); C2/=C2.max()
    a=np.ones(len(x))/len(x); b=np.ones(len(y))/len(y)
    G0=np.eye(len(x),len(y))/len(x)
    T=ot.gromov.gromov_wasserstein(C1,C2,a,b,'square_loss',G0=G0,epsilon=2e-3,max_iter=400,tol=1e-10,verbose=False)
    return T,C1,C2
def draw_match(x,y,T,filename, color=VIOLET, distort=None):
    xd=x+np.array([-1.22,0]); yd=y+np.array([1.22,0])
    pairs=[(i,int(np.argmax(T[i])),float(T[i].max())) for i in range(T.shape[0])]
    pts=np.vstack([xd,yd]); xlim,ylim=padded_limits(pts,pad=0.10)
    fig,ax=plt.subplots(figsize=(2.72,1.75))
    if distort is None:
        draw_transport_segments(ax,xd,yd,pairs,color=color,max_width=1.10,alpha_scale=0.48,zorder=1)
    else:
        segs=[]; cols=[]; widths=[]
        dd=distort/max(distort.max(),1e-12)
        for i,j,m in pairs:
            segs.append([xd[i],yd[j]]); cols.append((*interp_color(float(0.15+0.75*dd[i]), VIOLET, ORANGE),0.34+0.42*dd[i])); widths.append(0.25+1.05*np.sqrt(dd[i]+0.04))
        ax.add_collection(LineCollection(segs,colors=cols,linewidths=widths,zorder=1))
    draw_point_clouds(ax,xd,yd,base_size=DIRAC_MARKER_SIZE*0.88)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal'); remove_axes(ax)
    save_pdf(fig,out/filename,pad_inches=0.045); plt.close(fig)

x=base_shape(16)
y=rotate(x,0.64); y[:,0]*=1.28+0.18*(np.arange(len(y))>8); y[:,1]*=0.78; y[9:,1]-=0.25
T,C1,C2=gw_plan(x,y); match=np.argmax(T,axis=1)
res=np.abs(C1-C2[np.ix_(match,match)])
point=res.mean(axis=1)
draw_match(x,y,T,'correspondence.pdf',distort=point)
fig,ax=plt.subplots(figsize=(1.75,1.75))
im=ax.imshow(res,cmap='magma',interpolation='nearest')
ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values(): sp.set_visible(True); sp.set_linewidth(0.75); sp.set_color('#333333')
save_pdf(fig,out/'distance-residual.pdf',pad_inches=0.045); plt.close(fig)